# Experimentos do artigo (Colab, GPU T4)

**Antes de rodar:** Ambiente de execução → Alterar o tipo → **GPU T4**. Depois: **Ambiente de execução → Executar tudo**.

O que este notebook faz (~1 h 30 no total):
1. **H1/H2:** treina o BERTimbau com **rótulos humanos** (H) e com **rótulos do LLM** (L) nos mesmos textos do RePro, 3 sementes cada, e um controle só com a validação humana (V).
2. **H3:** aplica o modelo humano à Olist (metade "relato") e mede se "entrega negativa" acompanha o atraso real.
3. Salva tudo no Google Drive (`MyDrive/pln-absa-olist/resultados/`). **Se a sessão cair, rode de novo: o que já terminou é pulado.**
4. No fim, baixa `resultados.zip`, com as predições e os registros usados nas tabelas do artigo.

In [ ]:
# 1) Repositório, pacote e Drive
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd /content/pln-absa-olist
    !pip -q install -e ".[transformers]"
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        SAIDA = "/content/drive/MyDrive/pln-absa-olist/resultados"
    except Exception as erro:  # permissão negada ou parcial: segue sem o Drive
        print(f"⚠️ Drive não conectou ({erro}). Os resultados ficam só nesta sessão:")
        print("   NÃO feche a aba até o resultados.zip ser baixado no passo 6.")
        SAIDA = "/content/resultados"
else:
    SAIDA = "results/encoders"
from pathlib import Path
# A instalação editável só vale depois de reiniciar o Python; isto torna o pacote visível já nesta sessão
_src = str(Path("src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
SAIDA = Path(SAIDA); SAIDA.mkdir(parents=True, exist_ok=True)
MODELOS = SAIDA.parent / "modelos"; MODELOS.mkdir(parents=True, exist_ok=True)  # fora do zip de resultados
CKPT = Path("/content/ckpt" if "google.colab" in sys.modules else SAIDA.parent / "ckpt")  # disco local, rápido
CKPT.mkdir(parents=True, exist_ok=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o ambiente para GPU T4!")

In [ ]:
# 2) Dados: RePro (gabarito humano) e rótulos do LLM nos mesmos textos
import json, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from absa.config import ASPECTS, LABEL_COLUMNS, POLARITIES, ROOT, SEED
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1, macro_f1, detection_f1
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light
from absa.repro import load_repro

treino, val, teste, _ = load_repro()
demo, val_dev = train_test_split(val, train_size=200, stratify=val.nota, random_state=SEED)
demo, val_dev = demo.reset_index(drop=True), val_dev.reset_index(drop=True)

arq_llm = sorted((ROOT / "results" / "llm").glob("repro_treino_*.csv"))
if arq_llm:
    llm = pd.read_csv(arq_llm[0])
    llm = llm[llm.valido]
    treino_llm = treino[["review_id", "texto_modelo"]].merge(llm[["review_id", *LABEL_COLUMNS]], on="review_id")
    print(f"rótulos do LLM: {arq_llm[0].name} — {len(treino_llm)} textos válidos")
else:
    treino_llm = None
    print("Sem rótulos do LLM no repositório: a condição L será pulada.")

entrada = lambda df: df.texto_modelo.map(clean_light).tolist()
print(f"treino {len(treino)} · val_dev {len(val_dev)} · demo {len(demo)} · teste {len(teste)}")

In [ ]:
# 3) Condições e sementes (prioridade: H e L com 3 sementes; V com 1)
SEEDS = [13, 42, 87]
CONDICOES = [("H", s) for s in SEEDS]
if treino_llm is not None:
    CONDICOES += [("L", s) for s in SEEDS]
CONDICOES += [("V", 42)]

def dados(cond):
    if cond == "H":
        return entrada(treino), treino, entrada(val_dev), val_dev
    if cond == "L":
        return entrada(treino_llm), treino_llm, entrada(val_dev), val_dev
    return entrada(val_dev), val_dev, entrada(demo), demo  # V: só validação humana; época escolhida no demo

def treinar(cond, seed):
    pred_arq = SAIDA / f"pred_teste_bertimbau_{cond}_s{seed}.csv"
    if pred_arq.exists():
        print(f"✓ {cond} s{seed} já feito — pulando"); return
    X, y, Xd, yd = dados(cond)
    cfg = TrainConfig(model_name=ENCODERS["bertimbau"], epochs=4, lr=3e-5, batch_size=32, max_len=128, seed=seed)
    clf = EncoderAspectClassifier(cfg)
    t0 = time.time()
    ckpt = CKPT / f"{cond}_s{seed}.pt"
    clf.fit(X, y, Xd, yd, save_path=ckpt)
    clf.load(ckpt)
    proba = clf.predict_proba(entrada(teste))
    pred = pd.DataFrame({c: [POLARITIES[i] for i in proba[:, j].argmax(1)] for j, c in enumerate(LABEL_COLUMNS)})
    pred.insert(0, "review_id", teste.review_id.values)
    pred.to_csv(pred_arq, index=False)
    json.dump({"cond": cond, "seed": seed, "n_treino": len(X), "minutos": (time.time() - t0) / 60,
               "historico": clf.history}, open(SAIDA / f"log_bertimbau_{cond}_s{seed}.json", "w"), indent=1)
    if cond == "H" and seed == 42:
        torch.save(clf.model.state_dict(), MODELOS / "bertimbau_H_s42.pt")  # usado na Olist (passo 5)
    print(f"→ {cond} s{seed}: F1 de pares {tuple_f1(teste, pred):.3f} ({(time.time() - t0) / 60:.1f} min)")
    ckpt.unlink(missing_ok=True)
    del clf; torch.cuda.empty_cache()

for cond, seed in CONDICOES:
    treinar(cond, seed)

In [ ]:
# 4) Resumo no teste humano (média ± desvio entre sementes)
linhas = []
for arq in sorted(SAIDA.glob("pred_teste_bertimbau_*.csv")):
    p = pd.read_csv(arq).set_index("review_id").loc[teste.review_id, LABEL_COLUMNS].reset_index(drop=True)
    cond, seed = arq.stem.split("_")[-2], arq.stem.split("_")[-1]
    linhas.append({"cond": cond, "seed": seed, "f1_pares": tuple_f1(teste, p),
                   "f1_macro": macro_f1(teste, p), "f1_deteccao": detection_f1(teste, p)})
res = pd.DataFrame(linhas)
display(res.groupby("cond")[["f1_pares", "f1_macro", "f1_deteccao"]].agg(["mean", "std"]).round(3))

In [ ]:
# 5) H3 — Olist (metade "relato"): P(entrega negativa) × atraso real
from sklearn.metrics import roc_auc_score
from absa.data import load_reviews
from absa.sampling import gold_exclusion, olist_split

arq_olist = SAIDA / "olist_relato_probs_bertimbau_H_s42.csv"
if not arq_olist.exists():
    olist = load_reviews()
    olist = olist[olist.entregue & (olist_split(olist) == "relato") & ~gold_exclusion(olist)].reset_index(drop=True)
    clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS["bertimbau"], max_len=128, batch_size=64))
    clf.model.load_state_dict(torch.load(MODELOS / "bertimbau_H_s42.pt", map_location=clf.device))
    proba = clf.predict_proba(entrada(olist))
    out = olist[["review_id", "order_id", "nota", "atrasado", "atraso_dias", "regiao", "categoria"]].copy()
    for j, a in enumerate(ASPECTS):
        for k, pol in enumerate(POLARITIES):
            out[f"p_{a}_{pol}"] = proba[:, j, k]
    out.to_csv(arq_olist, index=False)
olist_p = pd.read_csv(arq_olist)
y = olist_p.atrasado.values
print(f"n = {len(olist_p)}, atrasados = {y.sum()}")
print("AUC P(entrega neg):", round(roc_auc_score(y, olist_p.p_entrega_negativo), 3),
      "| nota:", round(roc_auc_score(y, -olist_p.nota), 3),
      "| placebo P(produto neg):", round(roc_auc_score(y, olist_p.p_produto_negativo), 3))

In [ ]:
# 6) Baixar os resultados (resultados.zip)
import shutil
shutil.make_archive("/content/resultados", "zip", SAIDA)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/content/resultados.zip")